# Data Sourcing Armenpflege

Copy for secondary queryks

## Housekeeping

In [1]:
%pip install impresso

/home/bode-wsl/projects/summerschool26/.venv/bin/python: No module named pip
Note: you may need to restart the kernel to use updated packages.


❗ Please restart the kernel after installing the package to ensure that all changes take effect.

In [2]:
import os
import random
import time

import impresso
import pandas as pd

In [3]:
IN_COLAB = 'COLAB_RELEASE_TAG' in os.environ
DATA_DIR = '/content/drive/MyDrive/data' if IN_COLAB else '../data'

if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')

## Connect with the impresso API

In [4]:
client = impresso.connect()


Click on the following link to access the login page: https://impresso-project.ch/datalab/token
 - 🔤 Enter your email/password on this page.
 - 🔑 Once logged in, a secret token will be generated for you.
 - 📋 Copy this token and paste it into the input field below. Then press "Enter". 👇🏼.

🎉 You are now connected to the Impresso API!  🎉


In [11]:
# ⬇️⬇️⬇️ Your search query here

QUERY_PARAMS = {
    "with_text_contents": True,
    "language": "de",
    "term": "Armenpflege",
}

QUERY_NAME = "armenpflege_2"
# ⬆️⬆️⬆️ Your search query here 

In [12]:
search_results = client.search.find(
	**QUERY_PARAMS,
)
search_results

,issueId,relevanceScore,meta.sourceType,meta.sourceMedium,meta.date,meta.mediaId,meta.mediaTitle,meta.partnerId,meta.partnerTitle,meta.countryCode,...,access.copyrightLabel,semanticEnrichments.mentions.persons,semanticEnrichments.mentions.locations,semanticEnrichments.mentions.organisations,semanticEnrichments.mentions.newsagencies,text.originalLangCode,semanticEnrichments.namedEntities.persons,semanticEnrichments.namedEntities.locations,semanticEnrichments.namedEntities.organisations,semanticEnrichments.namedEntities.newsagencies
id,,,,,,,,,,,,,,,,,,,,,
DTT-1971-10-26-a-i0067,DTT-1971-10-26-a,0.736014,newspaper,print,1971-10-26T00:00:00+00:00,DTT,Die Tat,Migros,Migros,CH,...,Protected domain: in copyright,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
DTT-1958-03-20-a-i0036,DTT-1958-03-20-a,0.666813,newspaper,print,1958-03-20T00:00:00+00:00,DTT,Die Tat,Migros,Migros,CH,...,Protected domain: in copyright,[{'surfaceForm': 'Mitinhaber der Seidenweberei...,"[{'surfaceForm': 'Mainz', 'mentionConfidence':...",[],[],NaN,NaN,NaN,NaN,NaN
DTT-1956-10-15-a-i0094,DTT-1956-10-15-a,0.733817,newspaper,print,1956-10-15T00:00:00+00:00,DTT,Die Tat,Migros,Migros,CH,...,Protected domain: in copyright,"[{'surfaceForm': 'Frau Haas', 'mentionConfiden...",[],[],[],NaN,NaN,NaN,NaN,NaN


In [13]:
# Define the total amount of items you want to retrieve 
TOTAL_RESULTS = search_results.total
LIMIT = 1000 # Maximum is 1000

all_results = []
n_retrieved = 0

for offset in range(0, TOTAL_RESULTS, LIMIT):
    print(f"Retrieved {n_retrieved} / {TOTAL_RESULTS} items so far...", end="\r")

    search_results = client.search.find(
        **QUERY_PARAMS,
        limit=LIMIT,
        offset=offset
    )

    all_results.append(search_results.df)
    n_retrieved += len(search_results.df)

    print(f"Retrieved {n_retrieved} / {TOTAL_RESULTS} items so far...", end="\r")

# To conclude, you transform your list into a Pandas Dataframe and visualise it by running 'full_results_df'
full_results_df: pd.DataFrame = pd.concat(all_results) #, ignore_index=True)

# Save only unique index values
full_results_df = full_results_df[~full_results_df.index.duplicated(keep='first')]
full_results_df.to_csv(os.path.join(DATA_DIR, f"{QUERY_NAME}_index.csv"), index=True)

## Download the full text for all found hits

In [8]:
TIMEOUT = 2 # Be nice to the server and avoid sending too many requests in a short time

In [9]:
OUTPATH = os.path.join(DATA_DIR, f"{QUERY_NAME}.csv")
INDEXPATH = os.path.join(DATA_DIR, f"{QUERY_NAME}_index.csv")

try:
    index_df = pd.read_csv(INDEXPATH, index_col="id")
    wanted_ids = set(index_df.index.to_list())
except FileNotFoundError:
    wanted_ids = set()

try:
    fulltext_df = pd.read_csv(OUTPATH, index_col="id")
    retrieved_ids = set(fulltext_df.index.to_list())
except FileNotFoundError:
    fulltext_df = pd.DataFrame()
    retrieved_ids = set()

missing_ids = [id for id in wanted_ids if id not in retrieved_ids]

print(f"Missing {len(missing_ids)} items. (Already retrieved {len(retrieved_ids)})")
print(f"Requests for missing items will take approximately {len(missing_ids) * TIMEOUT / 60:.2f} minutes (assuming ~{1 / TIMEOUT:.2f} request per second).")

Missing 6872 items. (Already retrieved 101)
Requests for missing items will take approximately 229.07 minutes (assuming ~0.50 request per second).


Retrieve all missing documents

In [ ]:
try:
    for i, doc_id in enumerate(missing_ids):
        time.sleep(random.uniform(TIMEOUT * 0.5, TIMEOUT * 1.5))
        print(f"({i+1}/{len(missing_ids)}) - Processing document {doc_id}", end="\r")
        content_item = client.content_items.get(doc_id)
        content_item_df = content_item.df
        fulltext_df = pd.concat([fulltext_df, content_item_df])
        
        if (i + 1) % 100 == 0: # Save progress every 100 items
            fulltext_df.to_csv(OUTPATH, index=False)
except KeyboardInterrupt:
    print(f"Process interrupted by user. Saving progress...                              ")
    fulltext_df.to_csv(OUTPATH, index=False)
except Exception as e:
    print(f"An error occurred. Saving progress...                              ")
    fulltext_df.to_csv(OUTPATH, index=False)
    raise e
fulltext_df.to_csv(OUTPATH, index=False)

ERROR:root:Received error response (418): {"type":"https://impresso-project.ch/probs/unauthorized","title":"Downstream error","status":418,"detail":"SolrError: Unknown Solr error: "}


## Wrangle the data

In [7]:
fulltext_df[["meta.date", "meta.countryCode", "meta.provinceCode", "meta.mediaTitle", "text.langCode", "text.content"]].to_csv(os.path.join(DATA_DIR, "knappheit_reduced.csv"))
fulltext_df[["meta.date", "meta.countryCode", "meta.provinceCode", "meta.mediaTitle", "text.langCode", "text.content"]].to_parquet(os.path.join(DATA_DIR, "knappheit_reduced.parquet"))

In [9]:
fulltext_df[["id", "meta.date", "meta.mediaTitle", "text.itemTypeLabel", "text.content", "text.contentLength"]]

,id,meta.date,meta.mediaTitle,text.itemTypeLabel,text.content,text.contentLength
0,NZZ-1916-06-28-a-i0002,1916-06-28T00:00:00+00:00,Neue Zürcher Zeitung,Page,KOn « öß5 2s. Ml » 7? l Wte Mmyemeu » e!» r >;...,2630
0,NZZ-1854-07-01-a-i0001,1854-07-01T00:00:00+00:00,Neue Zürcher Zeitung,Page,"md « oran »» e,« hl « n «: »«« lt «»«,««»? « r...",1168
0,NZZ-1914-05-26-c-i0001,1914-05-26T00:00:00+00:00,Neue Zürcher Zeitung,Page,"Iellnng 133. InhlMg. Dienstag, 26. Mai 1914. 8...",2963
0,NZZ-1876-01-11-a-i0003,1876-01-11T00:00:00+00:00,Neue Zürcher Zeitung,Page,"Letzten, haben entved « ihr ganzes Lebn. in Ba...",2862
0,NZZ-1916-09-12-d-i0001,1916-09-12T00:00:00+00:00,Neue Zürcher Zeitung,Page,1444. Zweite » MittaMatt. 3tt Mchtt IMlNlg 137...,3309
...,...,...,...,...,...,...
0,handelsztg-1874-01-30-a-i0001,1874-01-30T00:00:00+00:00,Schweizerische Handels-Zeitung,Page,"■. J'JIM «I. ""TB—— -TO»»WI»WWIW>»liWW»l>»LWM l...",664
0,NZZ-1906-09-12-b-i0001,1906-09-12T00:00:00+00:00,Neue Zürcher Zeitung,Page,353 Zweites Morgenblatt. Per Zürcher Ubonnemen...,3605
0,NZZ-1929-11-28-f-i0001,1929-11-28T00:00:00+00:00,Neue Zürcher Zeitung,Page,Donnerstag 28. November » 929 Nlatt Neue Zürch...,3398
0,NZZ-1832-06-23-a-i0001,1832-06-23T00:00:00+00:00,Neue Zürcher Zeitung,Page,Plinumeratlon. Mbrlich « « e. H « NiiablUch 4 ...,1358
